# Session 2 — A/B Testing & Canary Deployments

**Goal:** Understand how to safely roll out new model versions using traffic splitting.

**Canary deployment**: Send a small % of traffic to the new model before full rollout.
- Risk: If the canary has issues, only 10% of users are affected
- Monitoring window: Watch metrics for 24h before promoting to 100%

**A/B test**: Compare two model versions on business metrics (not just offline F1).
- Split: 50/50 traffic to version A and version B
- Measure: Which version drives better customer retention outcomes?

> **The retrain pipeline you triggered in the previous notebook already performed a 90/10 canary
> split automatically.** In this notebook you'll inspect that live state and review the patterns
> for promoting to 100% or rolling back.

In [0]:
%run ../utils/config

In [0]:

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

endpoint_name = f"churn_{safe_username}"
model_name    = f"{catalog}.{schema}.churn_classifier"

# Check what versions we have
from mlflow import MlflowClient
mlflow_client = MlflowClient()
mlflow_client.set_registry_uri = lambda x: None

import mlflow
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
versions = client.search_model_versions(f"name='{model_name}'")
print(f"Available versions for {model_name}:")
for v in versions:
    # search_model_versions doesn't include aliases; use get_model_version instead
    mv = client.get_model_version(model_name, v.version)
    alias_list = mv.aliases if not callable(mv.aliases) else []
    aliases = ", ".join(alias_list) if alias_list else "(none)"
    print(f"  v{v.version}  aliases=[{aliases}]  status={v.status}")

## Live Endpoint State

The cell below reads the current endpoint configuration and shows the active traffic split.
If the retrain job has completed, you'll see the 90/10 canary split the pipeline applied.

In [0]:
import json

endpoint = w.serving_endpoints.get(endpoint_name)
print(json.dumps(endpoint.config.traffic_config.as_dict(), indent=2))

## What Happens Next: Promote or Roll Back

The pipeline left the endpoint in a 90/10 state. Your next decision as an MLOps engineer is
whether to promote the canary to full traffic or roll it back. Both operations are one step
in the Databricks UI:

**To promote to 100% (canary metrics look healthy):**
1. Go to **Serving** in the left sidebar
2. Click your <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">churn_{safe_username}</span> endpoint
3. Click **Edit configuration**
4. Set the new version to `100%` and remove the old version
5. Click **Update**

**To roll back (canary metrics degraded):**
1. Same path — **Edit configuration**
2. Set the old version back to `100%` and remove the canary version
3. Click **Update**

> **Rollback is instantaneous** — no redeployment, no downtime. Traffic shifts immediately.

**Optional — promote to 50/50 for a true A/B test:**
1. Set both versions to `50%` in Edit configuration
2. Let it run for 24-48 hours and compare business metrics across versions

| Decision | Action | Traffic after |
|---|---|---|
| Canary metrics healthy | Set new version = 100%, remove old | 100% new |
| Canary metrics degraded | Set old version = 100%, remove new | 100% old |
| Want A/B data | Set both versions = 50% | 50/50 |

> **Rollback is instantaneous** — just shift all traffic back to v1 and update. No redeployment needed.

## Discussion

- **How long should you monitor the canary before promoting to 100%?** (Depends on traffic volume — you need enough requests for statistical significance)
- **What business metrics would you track alongside PSI drift?** (Conversion rate, revenue per prediction, customer complaint rate)
- **Who approves the final promotion from 10% → 100%?** (Data scientist? MLOps engineer? Product owner?)



➡️ Next: [07_incident_runbook.ipynb]($./07_incident_runbook) *(optional bonus)* — walk through a full incident response